# Primeira Theory Revision exploratória — E4

Este notebook implementa o protocolo pré-registrado em `docs/protocolo_theory_revision_e4.md`. Ele preserva o notebook v2 e aplica C0, C1 e C2 como regras de construção a grafos de entrada independentes. Não executa o DptOIE, não altera Java e não usa solver.


In [1]:
from dataclasses import dataclass, field
from copy import deepcopy
from pathlib import Path
import hashlib
import json
import platform

import networkx as nx
import openpyxl

PROTOCOL_COMMIT = "79e77b8"
SOURCE_HEAD = "6bde4e276c0a46d1903eb733725bff81f4f5f525"
SPREADSHEET = Path("research/experimento/semana5_experimento_OIEC_PT_GOLD (3)(1).xlsx")
PARSE_PATHS = {
    "S009": Path("logs/s009/reproducao_instrumentada_trace_on/output/output_DP.conll"),
    "S003": Path("logs/e4/controles_baseline/s003/output/output_DP.conll"),
    "S005": Path("logs/e4/controles_baseline/s005/output/output_DP.conll"),
    "S010": Path("logs/e4/controles_baseline/s010/output/output_DP.conll"),
}
EXPECTED_HASHES = {
    str(SPREADSHEET): "a00c037da53f5de90307d8918b8547d226888297eb7243d0272c68d5f337616c",
    str(PARSE_PATHS["S009"]): "e94d708cef67efaa363a7fdb9e646e4fa39bc3c379e871923bf3631e96e2967f",
    str(PARSE_PATHS["S003"]): "f12731b7b6dc71ffda49d837f947f8eb76119751a4062585c8e564e3d0cb624c",
    str(PARSE_PATHS["S005"]): "49335c1eb391b21f55b6c44a956bfbfcc15ae968e75eb511f364e8a16a20ec2d",
    str(PARSE_PATHS["S010"]): "c3aaab2679596c96ca24e51896134ef572d7f963946c0af5c16ae50c4f4e5cfe",
}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

OBSERVED_HASHES = {path: sha256(path) for path in EXPECTED_HASHES}
assert OBSERVED_HASHES == EXPECTED_HASHES

ENVIRONMENT = {
    "python": platform.python_version(),
    "networkx": nx.__version__,
    "openpyxl": openpyxl.__version__,
    "protocol_commit": PROTOCOL_COMMIT,
    "source_head_before_protocol": SOURCE_HEAD,
}
print(json.dumps({"environment": ENVIRONMENT, "input_hashes": OBSERVED_HASHES}, ensure_ascii=False, indent=2))


{
  "environment": {
    "python": "3.12.0",
    "networkx": "3.5",
    "openpyxl": "3.1.5",
    "protocol_commit": "79e77b8",
    "source_head_before_protocol": "6bde4e276c0a46d1903eb733725bff81f4f5f525"
  },
  "input_hashes": {
    "research\\experimento\\semana5_experimento_OIEC_PT_GOLD (3)(1).xlsx": "a00c037da53f5de90307d8918b8547d226888297eb7243d0272c68d5f337616c",
    "logs\\s009\\reproducao_instrumentada_trace_on\\output\\output_DP.conll": "e94d708cef67efaa363a7fdb9e646e4fa39bc3c379e871923bf3631e96e2967f",
    "logs\\e4\\controles_baseline\\s003\\output\\output_DP.conll": "f12731b7b6dc71ffda49d837f947f8eb76119751a4062585c8e564e3d0cb624c",
    "logs\\e4\\controles_baseline\\s005\\output\\output_DP.conll": "49335c1eb391b21f55b6c44a956bfbfcc15ae968e75eb511f364e8a16a20ec2d",
    "logs\\e4\\controles_baseline\\s010\\output\\output_DP.conll": "c3aaab2679596c96ca24e51896134ef572d7f963946c0af5c16ae50c4f4e5cfe"
  }
}


## 1. Dados reais e validação de origem

As linhas A41–A44, A20, A28, A51 e A52 são lidas da aba `analise`. Tokens e dependências são lidos exclusivamente dos CoNLL preservados.


In [2]:
def load_conll(path):
    tokens = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        token_id = int(fields[0])
        tokens[token_id] = {
            "id": token_id,
            "form": fields[1],
            "pos": fields[3],
            "head": int(fields[6]),
            "deprel": fields[7],
            "ordem": token_id,
        }
    return tokens

PARSES = {case: load_conll(path) for case, path in PARSE_PATHS.items()}

workbook = openpyxl.load_workbook(SPREADSHEET, read_only=True, data_only=True)
sheet = workbook["analise"]
raw_rows = list(sheet.iter_rows(values_only=True))
headers = raw_rows[0]
ANALYSIS_ROWS = {
    f"A{excel_row}": dict(zip(headers, values))
    for excel_row, values in enumerate(raw_rows[1:], start=2)
}
SELECTED_ROWS = ("A41", "A42", "A43", "A44", "A28", "A51", "A52", "A20")
for row_id in SELECTED_ROWS:
    assert ANALYSIS_ROWS[row_id]["extrator"] == "DPTOIE"

print("SELECTED DATA")
for row_id in SELECTED_ROWS:
    row = ANALYSIS_ROWS[row_id]
    print(row_id, row["id_sentenca"], row["classificacao"], row["tripla_extraida"])


SELECTED DATA
A41 S009 E- (Catalano, voltou, para a área)
A42 S009 E- (Catalano, voltou, para a área depois de viver em Londres)
A43 S009 E- (Catalano, voltou, para a área depois de viver em Londres e em Glasgow , onde estudou design de jóias)
A44 S009 E- (Catalano, voltou, para a área depois de viver em Londres e em Glasgow , onde estudou design de jóias en a Glasgow School of Art)
A28 S005 E+ (alguém, tem, dignidade para continuar a protestar contra esse fato)
A51 S010 E+ (Eu, visitei, os espetáculos universitários en a Universidade de Northampton)
A52 S010 E+ (Eu, visitei, os espetáculos universitários en a Universidade de Northampton para encontrar novos talentos)
A20 S003 E+ (isso, é, cada vez mais digno de atenção)


## 2. Modelo de grafo, estados de regência e candidatos

`StatusRege` é fornecido manualmente e permanece separado das dependências. C0 ignora regência e rótulos; C1 exige `case+nmod`; C2 usa o estado trivalorado e possui ramo conservador.


In [3]:
TRUE_PROVISIONAL = "TRUE_PROVISIONAL"
FALSE_PROVISIONAL = "FALSE_PROVISIONAL"
INDETERMINATE = "INDETERMINATE"
DEPENDENCIA = "DEPENDENCIA"
OPENIE = "OPENIE"
CONSTRUCAO = "CONSTRUCAO"

@dataclass(frozen=True)
class TargetPreposition:
    p: int
    n: int
    status_rege: str
    status_origin: str
    justification: str

@dataclass(frozen=True)
class ExperimentalCase:
    case_id: str
    sentence_id: str
    row_id: str
    kind: str
    arg0: tuple
    rel: tuple
    arg1: tuple
    expected_e4_arg0: tuple
    expected_e4_rel: tuple
    expected_e4_arg1: tuple
    gold_arg0: tuple
    gold_rel: tuple
    gold_arg1: tuple
    relation_core: int
    targets: tuple
    notes: str = ""

@dataclass(frozen=True)
class RevisionCandidate:
    name: str
    description: str
    complexity: tuple

CANDIDATES = {
    "C0": RevisionCandidate("C0", "transferência sem guarda de regência ou rótulo de dependência", (0, 2, 1, 0)),
    "C1": RevisionCandidate("C1", "transferência condicionada por case(p,n) e nmod(n,v)", (2, 2, 1, 0)),
    "C2": RevisionCandidate("C2", "divisão conservadora por estado explícito de regência", (1, 2, 1, 1)),
}

def complexity_record(candidate):
    labels = ("guard_edits", "effect_edits", "interval_edits", "rule_splits")
    values = candidate.complexity
    return {**dict(zip(labels, values)), "total": sum(values)}

print(json.dumps({name: {"description": c.description, "complexity": complexity_record(c)} for name, c in CANDIDATES.items()}, ensure_ascii=False, indent=2))


{
  "C0": {
    "description": "transferência sem guarda de regência ou rótulo de dependência",
    "complexity": {
      "guard_edits": 0,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 0,
      "total": 3
    }
  },
  "C1": {
    "description": "transferência condicionada por case(p,n) e nmod(n,v)",
    "complexity": {
      "guard_edits": 2,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 0,
      "total": 5
    }
  },
  "C2": {
    "description": "divisão conservadora por estado explícito de regência",
    "complexity": {
      "guard_edits": 1,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 1,
      "total": 5
    }
  }
}


## 3. Instâncias experimentais

Os spans são definidos por IDs reais e confrontados com os componentes da planilha. A43/A44 mantêm seus erros E3/S3 no estado esperado de E4.


In [4]:
MANUAL_ORIGIN = "protocolo_theory_revision_e4.md; classificação manual provisória apoiada por sentença, GOLD, parse e justificativa gramatical"

def target(p, n, status, justification):
    return TargetPreposition(p, n, status, MANUAL_ORIGIN, justification)

CASES = (
    ExperimentalCase("S009-A41", "S009", "A41", "E_MINUS_PRIMARY", (1,), (2,), (3,4,5), (1,), (2,3), (4,5), (1,), (2,3), (4,5), 2, (target(3,5,TRUE_PROVISIONAL,"voltar para; GOLD em REL; proveniência confirmada"),)),
    ExperimentalCase("S009-A42", "S009", "A42", "E_MINUS_VARIANT", (1,), (2,), tuple(range(3,11)), (1,), (2,3), tuple(range(4,11)), (1,), (2,3), tuple(range(4,11)), 2, (target(3,5,TRUE_PROVISIONAL,"mesmo par voltar–para da sentença S009"),)),
    ExperimentalCase("S009-A43", "S009", "A43", "E_MINUS_MIXED", (1,), (2,), tuple(range(3,20)), (1,), (2,3), tuple(range(4,20)), (1,), (2,3), tuple(range(4,11)), 2, (target(3,5,TRUE_PROVISIONAL,"mesmo par voltar–para; E3/S3 permanecem fora do alvo"),), "E4+E3+S3"),
    ExperimentalCase("S009-A44", "S009", "A44", "E_MINUS_MIXED", (1,), (2,), tuple(range(3,26)), (1,), (2,3), tuple(range(4,26)), (1,), (2,3), tuple(range(4,11)), 2, (target(3,5,TRUE_PROVISIONAL,"mesmo par voltar–para; E3/S3 permanecem fora do alvo"),), "E4+E3+S3"),
    ExperimentalCase("S005-A28", "S005", "A28", "CONTROL", (15,), (16,), tuple(range(17,25)), (15,), (16,), tuple(range(17,25)), (15,), (16,), tuple(range(17,25)), 16, (target(18,19,FALSE_PROVISIONAL,"para marca continuar; finalidade/conteúdo, não regência de tem"), target(22,24,FALSE_PROVISIONAL,"contra pertence ao predicado encaixado protestar"))),
    ExperimentalCase("S010-A51", "S010", "A51", "CONTROL", (1,), (2,), tuple(range(3,11)), (1,), (2,), tuple(range(3,11)), (1,), (2,), tuple(range(3,11)), 2, (target(6,8,FALSE_PROVISIONAL,"locativo na Universidade, não regência de visitei"),), "fronteiras GOLD iguais; superfície en a/em a"),
    ExperimentalCase("S010-A52", "S010", "A52", "CONTROL", (1,), (2,), tuple(range(3,15)), (1,), (2,), tuple(range(3,15)), (1,), (2,), tuple(range(3,15)), 2, (target(6,8,FALSE_PROVISIONAL,"locativo na Universidade, não regência de visitei"), target(11,12,FALSE_PROVISIONAL,"para marca encontrar; oração de finalidade")), "fronteiras GOLD iguais; superfície en a/em a"),
    ExperimentalCase("S003-A20", "S003", "A20", "CONTROL", (7,), (8,), tuple(range(9,15)), (7,), (8,), tuple(range(9,15)), (7,), (8,), tuple(range(9,15)), 8, (target(13,14,FALSE_PROVISIONAL,"de atenção complementa o adjetivo digno, não a cópula"),), "parse global discutível"),
)

def token_text(sentence_id, token_ids):
    return " ".join(PARSES[sentence_id][token_id]["form"] for token_id in token_ids)

def normalize_space(text):
    return " ".join(str(text).split())

for case in CASES:
    row = ANALYSIS_ROWS[case.row_id]
    assert row["id_sentenca"] == case.sentence_id
    assert normalize_space(token_text(case.sentence_id, case.arg0)) == normalize_space(row["arg0_extraido"])
    assert normalize_space(token_text(case.sentence_id, case.rel)) == normalize_space(row["rel_extraida"])
    assert normalize_space(token_text(case.sentence_id, case.arg1)) == normalize_space(row["arg1_extraido"])

print("CASE INVENTORY")
for case in CASES:
    statuses = [(PARSES[case.sentence_id][t.p]["form"], t.status_rege) for t in case.targets]
    print(case.case_id, case.kind, statuses, "ARG0=", token_text(case.sentence_id, case.arg0), "REL=", token_text(case.sentence_id, case.rel), "ARG1=", token_text(case.sentence_id, case.arg1))


CASE INVENTORY
S009-A41 E_MINUS_PRIMARY [('para', 'TRUE_PROVISIONAL')] ARG0= Catalano REL= voltou ARG1= para a área
S009-A42 E_MINUS_VARIANT [('para', 'TRUE_PROVISIONAL')] ARG0= Catalano REL= voltou ARG1= para a área depois de viver em Londres
S009-A43 E_MINUS_MIXED [('para', 'TRUE_PROVISIONAL')] ARG0= Catalano REL= voltou ARG1= para a área depois de viver em Londres e em Glasgow , onde estudou design de jóias
S009-A44 E_MINUS_MIXED [('para', 'TRUE_PROVISIONAL')] ARG0= Catalano REL= voltou ARG1= para a área depois de viver em Londres e em Glasgow , onde estudou design de jóias en a Glasgow School of Art
S005-A28 CONTROL [('para', 'FALSE_PROVISIONAL'), ('contra', 'FALSE_PROVISIONAL')] ARG0= alguém REL= tem ARG1= dignidade para continuar a protestar contra esse fato
S010-A51 CONTROL [('en', 'FALSE_PROVISIONAL')] ARG0= Eu REL= visitei ARG1= os espetáculos universitários en a Universidade de Northampton
S010-A52 CONTROL [('en', 'FALSE_PROVISIONAL'), ('para', 'FALSE_PROVISIONAL')] ARG0= Eu 

## 4. Matching e aplicação sobre a construção

O grafo de entrada contém dependências, ARG0 e o candidato a núcleo, mas nenhuma aresta REL/ARG1. Os componentes resultantes são adicionados uma única vez após o matching; não há correção de um estado final.


In [5]:
def add_edge(graph, source, target_id, relation, layer):
    graph.add_edge(source, target_id, rel=relation, layer=layer)

def build_input_graph(case):
    graph = nx.DiGraph(name=f"G_input_{case.case_id}")
    graph.add_node("ROOT", type="Root")
    graph.add_node("e", type="Extraction", case_id=case.case_id)
    graph.add_node("construction", type="ConstructionContext")
    for token_id, data in PARSES[case.sentence_id].items():
        graph.add_node(token_id, type="Token", **data)
    for token_id, data in PARSES[case.sentence_id].items():
        head = data["head"] if data["head"] != 0 else "ROOT"
        add_edge(graph, token_id, head, data["deprel"], DEPENDENCIA)
    for token_id in case.arg0:
        add_edge(graph, "e", token_id, "ARG0", OPENIE)
    add_edge(graph, "construction", case.relation_core, "CANDIDATO_NUCLEO_REL", CONSTRUCAO)
    return graph

def component_ids(graph, component):
    ids = [target_id for source, target_id, data in graph.edges(data=True) if source == "e" and data.get("layer") == OPENIE and data.get("rel") == component]
    return tuple(sorted(ids, key=lambda token_id: graph.nodes[token_id].get("ordem", -1)))

def openie_state(graph):
    return {component: component_ids(graph, component) for component in ("ARG0", "REL", "ARG1")}

def canonical_graph_hash(graph):
    nodes = sorted((str(node), sorted((str(k), str(v)) for k, v in data.items())) for node, data in graph.nodes(data=True))
    edges = sorted((str(u), str(v), sorted((str(k), str(value)) for k, value in data.items())) for u, v, data in graph.edges(data=True))
    payload = json.dumps({"nodes": nodes, "edges": edges}, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def base_checks(case, target_item):
    tokens = PARSES[case.sentence_id]
    p, n, v = target_item.p, target_item.n, case.relation_core
    interval = tuple(token_id for token_id in sorted(tokens) if p <= token_id <= n)
    checks = {
        "p_in_argument": p in case.arg1,
        "n_child_of_v": tokens[n]["head"] == v,
        "p_child_of_n": tokens[p]["head"] == n,
        "p_before_n": tokens[p]["ordem"] < tokens[n]["ordem"],
        "interval_available": bool(interval),
    }
    return checks, interval

def match_target(case, target_item, candidate_name):
    tokens = PARSES[case.sentence_id]
    checks, interval = base_checks(case, target_item)
    base_match = all(checks.values())
    if candidate_name == "C0":
        matched = base_match
        branch = "C0_TRANSFER" if matched else "BASELINE_FALLBACK"
    elif candidate_name == "C1":
        checks["case_p_n"] = tokens[target_item.p]["deprel"] == "case"
        checks["nmod_n_v"] = tokens[target_item.n]["deprel"] == "nmod"
        matched = all(checks.values())
        branch = "C1_DEPENDENCY_TRANSFER" if matched else "BASELINE_FALLBACK"
    elif candidate_name == "C2":
        checks["status_true_provisional"] = target_item.status_rege == TRUE_PROVISIONAL
        matched = base_match and checks["status_true_provisional"]
        branch = "C2_TRUE_TRANSFER" if matched else "C2_CONSERVATIVE_DEFAULT"
    else:
        raise ValueError(candidate_name)
    return {"p": target_item.p, "n": target_item.n, "status_rege": target_item.status_rege, "status_origin": target_item.status_origin, "justification": target_item.justification, "checks": checks, "interval": interval, "matched": matched, "branch": branch}

def apply_construction(case, candidate_name):
    input_graph = build_input_graph(case)
    assert component_ids(input_graph, "REL") == tuple()
    assert component_ids(input_graph, "ARG1") == tuple()
    matches = [match_target(case, target_item, candidate_name) for target_item in case.targets]
    transferred = {record["p"] for record in matches if record["matched"]}
    result = deepcopy(input_graph)
    result.graph["candidate"] = candidate_name
    result.graph["matched_targets"] = tuple(sorted(transferred))
    for token_id in case.rel:
        add_edge(result, "e", token_id, "REL", OPENIE)
    for token_id in sorted(transferred):
        add_edge(result, "e", token_id, "REL", OPENIE)
    for token_id in case.arg1:
        if token_id not in transferred:
            add_edge(result, "e", token_id, "ARG1", OPENIE)
    return input_graph, result, matches


## 5. Execução dos 24 testes e matriz de resultados

Cada candidato é aplicado a uma cópia independente de cada um dos oito grafos. A classificação é calculada apenas depois da construção.


In [6]:
def expected_state(case):
    return {"ARG0": case.expected_e4_arg0, "REL": case.expected_e4_rel, "ARG1": case.expected_e4_arg1}

def baseline_state(case):
    return {"ARG0": case.arg0, "REL": case.rel, "ARG1": case.arg1}

def gold_state(case):
    return {"ARG0": case.gold_arg0, "REL": case.gold_rel, "ARG1": case.gold_arg1}

def component_texts(case, state):
    return {component: token_text(case.sentence_id, ids) for component, ids in state.items()}

def state_diff(left, right):
    return {component: {"removed": tuple(token_id for token_id in left[component] if token_id not in right[component]), "added": tuple(token_id for token_id in right[component] if token_id not in left[component])} for component in left}

RESULTS = []
for candidate_name in CANDIDATES:
    for case in CASES:
        input_graph, result_graph, matches = apply_construction(case, candidate_name)
        actual = openie_state(result_graph)
        expected = expected_state(case) if case.kind.startswith("E_MINUS") else baseline_state(case)
        if actual == expected:
            classification = "PASS_EXPLORATORY"
        elif case.kind.startswith("E_MINUS"):
            classification = "FAIL_CORRECTION"
        else:
            classification = "FAIL_OVERAPPLICATION"
        result_record = {
            "candidate": candidate_name,
            "case_id": case.case_id,
            "kind": case.kind,
            "input_graph_hash": canonical_graph_hash(input_graph),
            "result_graph_hash": canonical_graph_hash(result_graph),
            "input_openie": openie_state(input_graph),
            "baseline": baseline_state(case),
            "result": actual,
            "result_text": component_texts(case, actual),
            "expected_e4": expected_state(case),
            "gold": gold_state(case),
            "diff_from_baseline": state_diff(baseline_state(case), actual),
            "diff_from_gold": state_diff(gold_state(case), actual),
            "gold_exact_by_token_boundary": actual == gold_state(case),
            "matches": matches,
            "transferred_tokens": tuple(sorted({record["p"] for record in matches if record["matched"]})),
            "complexity": complexity_record(CANDIDATES[candidate_name]),
            "classification": classification,
            "notes": case.notes,
        }
        RESULTS.append(result_record)

priority = {"FAIL_OVERAPPLICATION": 3, "FAIL_CORRECTION": 2, "INCONCLUSIVE": 1, "PASS_EXPLORATORY": 0}
GLOBAL_RESULTS = {}
for candidate_name in CANDIDATES:
    case_results = [record["classification"] for record in RESULTS if record["candidate"] == candidate_name]
    GLOBAL_RESULTS[candidate_name] = max(case_results, key=lambda value: priority[value])

print("TEST MATRIX")
for record in RESULTS:
    print(record["candidate"], record["case_id"], record["classification"], "transferred=", record["transferred_tokens"], "result=", record["result_text"])
print("GLOBAL RESULTS", GLOBAL_RESULTS)


TEST MATRIX
C0 S009-A41 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área'}
C0 S009-A42 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres'}
C0 S009-A43 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres e em Glasgow , onde estudou design de jóias'}
C0 S009-A44 PASS_EXPLORATORY transferred= (3,) result= {'ARG0': 'Catalano', 'REL': 'voltou para', 'ARG1': 'a área depois de viver em Londres e em Glasgow , onde estudou design de jóias en a Glasgow School of Art'}
C0 S005-A28 FAIL_OVERAPPLICATION transferred= (18,) result= {'ARG0': 'alguém', 'REL': 'tem para', 'ARG1': 'dignidade continuar a protestar contra esse fato'}
C0 S010-A51 FAIL_OVERAPPLICATION transferred= (6,) result= {'ARG0': 'Eu', 'REL': 'visitei en', 'ARG1': 'os espetáculos universitários a Universidade de Northampton'}
C0 S

## 6. Asserções, limites de cobertura e registro final

As asserções verificam integridade, independência dos grafos, comportamento calculado e cobertura real. Não é declarada preservação das 15 extrações porque apenas quatro delas foram modeladas como controles.


In [7]:
assert len(CASES) == 8
assert len(RESULTS) == 24
assert all(record["input_openie"]["REL"] == tuple() and record["input_openie"]["ARG1"] == tuple() for record in RESULTS)
assert all(record["input_openie"]["ARG0"] for record in RESULTS)

def get_result(candidate, case_id):
    return next(record for record in RESULTS if record["candidate"] == candidate and record["case_id"] == case_id)

for candidate_name in CANDIDATES:
    assert get_result(candidate_name, "S009-A41")["result"] == {"ARG0": (1,), "REL": (2,3), "ARG1": (4,5)}

assert GLOBAL_RESULTS == {"C0": "FAIL_OVERAPPLICATION", "C1": "FAIL_OVERAPPLICATION", "C2": "PASS_EXPLORATORY"}
assert get_result("C0", "S005-A28")["classification"] == "FAIL_OVERAPPLICATION"
assert get_result("C1", "S005-A28")["classification"] == "PASS_EXPLORATORY"
assert get_result("C1", "S010-A51")["classification"] == "FAIL_OVERAPPLICATION"
assert all(get_result("C2", case.case_id)["classification"] == "PASS_EXPLORATORY" for case in CASES)
assert all(get_result("C2", case.case_id)["transferred_tokens"] == tuple() for case in CASES if case.kind == "CONTROL")
assert all(3 in get_result("C2", case.case_id)["transferred_tokens"] for case in CASES if case.sentence_id == "S009")

MODELED_CONTROL_OUTPUTS = 4
TOTAL_CONTROL_BASELINE_OUTPUTS = 15
FULL_15_NON_REGRESSION = "INCONCLUSIVE_NOT_FULLY_MODELED"
assert MODELED_CONTROL_OUTPUTS < TOTAL_CONTROL_BASELINE_OUTPUTS

FINAL_RECORD = {
    "protocol_commit": PROTOCOL_COMMIT,
    "input_integrity": "PASS",
    "modeled_cases": [case.case_id for case in CASES],
    "tests_executed": len(RESULTS),
    "global_results": GLOBAL_RESULTS,
    "complexity": {name: complexity_record(candidate) for name, candidate in CANDIDATES.items()},
    "full_15_non_regression": FULL_15_NON_REGRESSION,
    "assertions": "PASS",
    "interpretation_limit": "C2 usa anotação manual provisória; não reconhece regência operacionalmente",
}
print(json.dumps(FINAL_RECORD, ensure_ascii=False, indent=2))


{
  "protocol_commit": "79e77b8",
  "input_integrity": "PASS",
  "modeled_cases": [
    "S009-A41",
    "S009-A42",
    "S009-A43",
    "S009-A44",
    "S005-A28",
    "S010-A51",
    "S010-A52",
    "S003-A20"
  ],
  "tests_executed": 24,
  "global_results": {
    "C0": "FAIL_OVERAPPLICATION",
    "C1": "FAIL_OVERAPPLICATION",
    "C2": "PASS_EXPLORATORY"
  },
  "complexity": {
    "C0": {
      "guard_edits": 0,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 0,
      "total": 3
    },
    "C1": {
      "guard_edits": 2,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 0,
      "total": 5
    },
    "C2": {
      "guard_edits": 1,
      "effect_edits": 2,
      "interval_edits": 1,
      "rule_splits": 1,
      "total": 5
    }
  },
  "full_15_non_regression": "INCONCLUSIVE_NOT_FULLY_MODELED",
  "assertions": "PASS",
  "interpretation_limit": "C2 usa anotação manual provisória; não reconhece regência operacionalmente"
}
